# Manual Inspection

**Phase A** — bad channel detection (requires Stage 1 outputs)  
**Phase B** — bad epoch rejection + ICA component selection (requires Stage 2 outputs)

Each phase loops over subjects and skips any that are already done,  
so you can close the notebook mid-way and re-run safely.

Run cells in order. Close each interactive plot when finished marking —  
results are saved to `preprocessing/manual/` automatically.

Pipeline order: preproc_stage1_auto.py -> Phase 1 -> preproc_stage2_auto.py -> Phase 2 -> preproc_stage3_auto.py 

In [ ]:
%matplotlib qt

import os, re, json
import numpy as np
import matplotlib.pyplot as plt
import mne

BASE   = "" # add project path here
OUT1   = os.path.join(BASE, "preprocessing", "stage1")
OUT2   = os.path.join(BASE, "preprocessing", "stage2")
BRIDGE = os.path.join(BASE, "preprocessing", "bridging")
MANUAL = os.path.join(BASE, "preprocessing", "manual")

# Discover subjects from stage-1 outputs
raw_files = sorted(
    [f for f in os.listdir(OUT1) if f.endswith("_raw.fif") and "hpfilt" not in f],
    key=lambda f: int(re.search(r"p(\d+)", f).group(1)),
)
subjects = [re.search(r"(p\d+)_raw", f).group(1) for f in raw_files]
print(f"Subjects: {subjects}")

## Phase A — Bad channel detection

For each subject:
1. Opens the HP-filtered raw in an interactive window
2. **Mark bad channels** by clicking on their traces; close the window when done
3. Saves `preprocessing/manual/{subj}_bad_info.json` with the bad channel list and bridged pairs

In [ ]:
for subj in subjects:
    bad_info_path = os.path.join(MANUAL, f"{subj}_bad_info.json")
    hp_path       = os.path.join(OUT1,   f"{subj}_hpfilt_raw.fif")
    bridge_path   = os.path.join(BRIDGE, f"{subj}_bridged_pairs.npy")

    if os.path.exists(bad_info_path):
        print(f"{subj}: bad_info.json already exists — skipping.")
        continue

    if not os.path.exists(hp_path):
        print(f"{subj}: stage-1 HP file not found — run preproc_stage1_auto.py first.")
        continue

    print(f"\n{'='*55}\n  {subj} — Phase A: bad channels\n{'='*55}")

    # Show bridging figure so you can refer to it alongside the raw plot
    fig_path = os.path.join(BRIDGE, f"{subj}_bridging.png")
    if os.path.exists(fig_path):
        img = plt.imread(fig_path)
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.imshow(img); ax.axis("off")
        ax.set_title(f"{subj} — Electrical Distance (reference only)")
        plt.show(block=False)

    bridged_idx = []
    if os.path.exists(bridge_path):
        bridged_idx = np.load(bridge_path, allow_pickle=True).tolist()
        if bridged_idx:
            print(f"  Auto-detected bridged pairs: {bridged_idx}")

    raw = mne.io.read_raw_fif(hp_path, preload=True, verbose=False)

    # Interactive plot — click traces to mark bad channels; close when done
    raw.plot(
        n_channels=len(raw.ch_names),
        block=True,
        title=f"{subj} — click traces to mark bad channels, then CLOSE",
    )

    # Convert bridged_idx to plain Python ints so json.dump can serialize them
    bridged_idx_json = [[int(i), int(j)] for i, j in bridged_idx] if bridged_idx else []

    bad_info = {
        "bad_channels": raw.info["bads"],
        "bridged_pairs": bridged_idx_json,
    }
    with open(bad_info_path, "w") as f:
        json.dump(bad_info, f, indent=2)

    print(f"  Saved → {bad_info_path}")
    print(f"  Bad channels:  {bad_info['bad_channels']}")
    print(f"  Bridged pairs: {bad_info['bridged_pairs']}")
    plt.close("all")

print("\nPhase A complete (or all subjects already done).")

## Phase B — Bad epoch rejection + ICA component selection

For each subject:
1. Opens the temporary epochs — **press space / click to mark bad epochs**; close when done
2. Opens the ICA component overview and sources — **click components to toggle exclusion**; close when done
3. Saves `bad_epochs.json`, `ica_exclude.json`, and appends noisy-epoch removals to `removal_log.json`
   so that stage3 can add a `trial_number` column to the final epoch metadata.

EOG-correlated components are suggested automatically as a starting point.

In [ ]:
for subj in subjects:
    bad_epo_path = os.path.join(MANUAL, f"{subj}_bad_epochs.json")
    ica_exc_path = os.path.join(MANUAL, f"{subj}_ica_exclude.json")
    temp_epo_path = os.path.join(OUT2, f"{subj}_temp-epo.fif")
    ica_path      = os.path.join(OUT2, f"{subj}_ica.fif")

    if os.path.exists(bad_epo_path) and os.path.exists(ica_exc_path):
        print(f"{subj}: Phase B already complete — skipping.")
        continue

    if not os.path.exists(temp_epo_path) or not os.path.exists(ica_path):
        print(f"{subj}: stage-2 files not found — run preproc_stage2_auto.py first.")
        continue

    print(f"\n{'='*55}\n  {subj} — Phase B: epochs + ICA\n{'='*55}")

    epochs = mne.read_epochs(temp_epo_path, preload=True, verbose=False)
    ica    = mne.preprocessing.read_ica(ica_path, verbose=False)

    # ── Bad epoch inspection ─────────────────────────────────────────────
    if not os.path.exists(bad_epo_path):
        # Save centering-event sample numbers BEFORE the plot drops anything
        orig_events = epochs.events.copy()

        epochs.plot(
            block=True,
            title=f"{subj} — press space to mark bad epochs, then CLOSE",
        )

        # Collect all dropped epochs from drop_log
        bad_idx          = [i for i, dl in enumerate(epochs.drop_log) if len(dl) > 0]
        # Store as stimulus sample-numbers — unambiguous across re-indexing
        bad_stim_samples = orig_events[bad_idx, 0].tolist()

        with open(bad_epo_path, "w") as f:
            json.dump({
                "bad_epoch_indices": bad_idx,
                "bad_stim_samples":  bad_stim_samples,
            }, f, indent=2)
        print(f"  Bad epochs:  {len(bad_idx)} → {bad_epo_path}")

        # Append to removal_log so stage3 has a full picture
        removal_log_path = os.path.join(MANUAL, f"{subj}_removal_log.json")
        trial_map_path   = os.path.join(MANUAL, f"{subj}_trial_map.json")
        if os.path.exists(removal_log_path) and os.path.exists(trial_map_path):
            with open(removal_log_path) as f: removal_log = json.load(f)
            with open(trial_map_path)   as f: trial_map   = json.load(f)
            for sample in bad_stim_samples:
                key = str(int(sample))
                if key in trial_map:
                    removal_log["removed"].append({
                        "trial_number": trial_map[key],
                        "reason":       "noisy_epoch",
                        "stim_sample":  int(sample),
                    })
            with open(removal_log_path, "w") as f:
                json.dump(removal_log, f, indent=2)
            print(f"  Removal log updated ({len(removal_log['removed'])} total removals)")
        else:
            print(f"  Warning: trial_map or removal_log not found — "
                  f"noisy-epoch trial numbers not logged. Re-run preproc_stage1_auto.py.")

    else:
        # Bad epochs already marked — apply them so ICA sees clean data
        with open(bad_epo_path) as f:
            bad_data = json.load(f)
        bad_samples = bad_data["bad_stim_samples"]
        drop_mask   = np.isin(epochs.events[:, 0], bad_samples)
        epochs.drop(np.where(drop_mask)[0], reason="USER", verbose=False)
        print(f"  Applied existing bad-epoch list ({drop_mask.sum()} epochs dropped).")

    # ── ICA inspection ───────────────────────────────────────────────────
    if not os.path.exists(ica_exc_path):
        # Load the stage-1 raw to use find_bads_eog (needs the EOG channels)
        raw_path = os.path.join(OUT1, f"{subj}_raw.fif")
        raw      = mne.io.read_raw_fif(raw_path, preload=False, verbose=False)

        # Auto-detect EOG components as a starting suggestion
        eog_idx, _ = ica.find_bads_eog(raw, verbose=False)
        if eog_idx:
            print(f"  Suggested EOG components: {eog_idx} (verify manually)")

        # Component topographies overview
        ica.plot_components(title=f"{subj} — component topographies")

        # Time-course plot — click components to toggle; close when done
        ica.plot_sources(
            epochs,
            block=True,
            title=f"{subj} — click components to exclude, then CLOSE",
        )

        with open(ica_exc_path, "w") as f:
            json.dump({"ica_exclude": ica.exclude}, f, indent=2)
        print(f"  ICA exclude: {ica.exclude} → {ica_exc_path}")

    plt.close("all")

print("\nPhase B complete (or all subjects already done).")